In [16]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import matplotlib
matplotlib.use('Agg')  # For saving without display
import random

# Define weather types and their symbols
weather_symbols = {
    'Sun': '☀️',
    'Rain': '☂',  # Umbrella symbol for rain
    'Snow': '❄️',
    'Cloud': '☁️'
}

# Define colors for each weather type (for background highlighting)
weather_colors = {
    'Sun': '#FFD700',    # Gold
    'Rain': '#4FC3F7',   # Light blue
    'Snow': '#E0E0E0',   # Light gray
    'Cloud': '#BDBDBD'   # Gray
}

def create_weather_sequence(length=20, seed=42):
    """Create a random sequence of weather types"""
    random.seed(seed)
    weathers = list(weather_symbols.keys())
    return [random.choice(weathers) for _ in range(length)]

def find_combinations(sequence, condition_pair, lag=1):
    """Find indices where specified weather combination occurs at given lag"""
    first_element_highlight = [False] * len(sequence)
    second_element_highlight = [False] * len(sequence)
    pair1, pair2 = condition_pair
    
    for i in range(len(sequence) - lag):
        if sequence[i] == pair1 and sequence[i + lag] == pair2:
            first_element_highlight[i] = True  # First element (cause)
            second_element_highlight[i + lag] = True  # Second element (effect)
    
    return first_element_highlight, second_element_highlight

def count_combinations(sequence, condition_pair, lag):
    """Count how many times the pattern occurs"""
    pair1, pair2 = condition_pair
    count = 0
    for i in range(len(sequence) - lag):
        if sequence[i] == pair1 and sequence[i + lag] == pair2:
            count += 1
    return count

def create_visualization(sequence, first_highlight, second_highlight, condition_pair, lag, output_file='weather_sequence.png'):
    """Create the visualization with two-color highlighting and probability formula on the right"""
    
    # Count pattern occurrences
    n_patterns = count_combinations(sequence, condition_pair, lag)
    n_possible = len(sequence) - lag
    probability = n_patterns / n_possible
    
    # Get symbols for notation
    symbol1 = weather_symbols[condition_pair[0]]
    symbol2 = weather_symbols[condition_pair[1]]
    
    fig, ax = plt.subplots(figsize=(16, 3.5))
    ax.set_xlim(-0.5, len(sequence) - 0.5)
    ax.set_ylim(-0.8, 0.8)
    
    # Draw each weather symbol with optional highlighting
    for i, weather in enumerate(sequence):
        # Background highlight based on role in pattern
        if first_highlight[i]:
            # First element (cause) - green highlight
            rect = plt.Rectangle((i - 0.4, -0.7), 0.8, 1.2, 
                                 facecolor='lightgreen', alpha=0.6, 
                                 edgecolor='green', linewidth=2)
            ax.add_patch(rect)
        elif second_highlight[i]:
            # Second element (effect) - light blue highlight
            rect = plt.Rectangle((i - 0.4, -0.7), 0.8, 1.2, 
                                 facecolor='lightblue', alpha=0.6, 
                                 edgecolor='blue', linewidth=2)
            ax.add_patch(rect)
        
        # Add weather symbol
        ax.text(i, 0, weather_symbols[weather], 
                fontsize=35, ha='center', va='center',
                bbox=dict(boxstyle='circle,pad=0.2', 
                         facecolor=weather_colors[weather], 
                         edgecolor='black', linewidth=1.5))
        
        # Add small label below
        ax.text(i, -0.55, weather, fontsize=8, ha='center', va='center',
               color='#333333', fontweight='bold')
        
        # Add position number
        ax.text(i, -0.75, str(i+1), fontsize=7, ha='center', va='center',
               color='#666666')
    
    # Add connecting lines for highlighted pairs at specified lag
    for i in range(len(sequence) - lag):
        if first_highlight[i] and second_highlight[i + lag]:
            # Draw an arrow connecting the pair
            ax.annotate('', xy=(i + lag, 0.35), xytext=(i, 0.35),
                       arrowprops=dict(arrowstyle='->', color='red', 
                                      lw=2, alpha=0.7))
            # Add lag label above the arrow
            mid_x = i + lag/2
            ax.text(mid_x, 0.5, f'lag={lag}', fontsize=8, ha='center',
                   color='red', fontweight='bold')
    
    # Add probability formula on the RIGHT side
    formula_text = f"$\\hat{{p}}(X_{{t}} = {symbol1},\\; X_{{t+{lag}}} = {symbol2}) = \\frac{{{n_patterns}}}{{{n_possible}}}$"
    ax.text(len(sequence) + 1.5, 0, formula_text, 
            fontsize=13, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', 
                     edgecolor='black', linewidth=1.5))
    
    # Customize the plot
    ax.set_xticks(range(len(sequence)))
    ax.set_xticklabels([])  # Remove x-tick labels since we have position numbers below icons
    ax.set_xlabel('Time Position (t)', fontsize=12, fontweight='bold', labelpad=10)
    ax.set_title(f"Weather Sequence with Highlighted '{condition_pair[0]} {symbol1} → {condition_pair[1]} {symbol2}' at Lag {lag}",
                fontsize=13, fontweight='bold', pad=20)
    
    # Remove y-axis and spines
    ax.set_yticks([])
    ax.spines['left'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    
    # Add a subtle grid line at y=0
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.3, zorder=0)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Graphic saved as '{output_file}'")
    print(f"   Probability: {n_patterns}/{n_possible} = {probability:.3f}")
    return output_file

def main():
    # Create weather sequence
    sequence = create_weather_sequence(length=20, seed=42)
    
    # Define the condition to highlight: Sun followed by Rain
    condition_pair = ('Sun', 'Rain')
    lag = 2  # Look at relationship with 2-step lag
    
    # Find which positions to highlight (separate arrays for first and second element)
    first_highlight, second_highlight = find_combinations(sequence, condition_pair, lag)
    
    # Count patterns
    n_patterns = count_combinations(sequence, condition_pair, lag)
    n_possible = len(sequence) - lag
    
    # Print the sequence information
    print("\n" + "="*60)
    print("Weather Sequence (positions 1-20):")
    print("="*60)
    for i, weather in enumerate(sequence, 1):
        symbol = weather_symbols[weather]
        if first_highlight[i-1]:
            highlight = "🟢 (cause)"
        elif second_highlight[i-1]:
            highlight = "🔵 (effect)"
        else:
            highlight = "   "
        print(f"Pos {i:2d}: {weather:5s} {symbol}  {highlight}")
    
    print(f"\n📊 Pattern Analysis:")
    print(f"   Looking for: '{condition_pair[0]} ☀️ → {condition_pair[1]} ☂' with lag = {lag}")
    print(f"   🟢 First elements (cause): {sum(first_highlight)} positions")
    print(f"   🔵 Second elements (effect): {sum(second_highlight)} positions")
    print(f"\n   Estimated probability:")
    print(f"   p̂ = {n_patterns}/{n_possible} = {n_patterns/n_possible:.3f}")
    
    # Create and save visualization
    output_file = 'weather_bivariate_pattern.png'
    create_visualization(sequence, first_highlight, second_highlight, condition_pair, lag, output_file)
    
    print(f"\n✅ File saved! You can now insert '{output_file}' into your presentation.")

if __name__ == "__main__":
    main()


Weather Sequence (positions 1-20):
Pos  1: Sun   ☀️     
Pos  2: Sun   ☀️  🟢 (cause)
Pos  3: Snow  ❄️     
Pos  4: Rain  ☂  🔵 (effect)
Pos  5: Rain  ☂     
Pos  6: Rain  ☂     
Pos  7: Sun   ☀️     
Pos  8: Sun   ☀️     
Pos  9: Cloud ☁️     
Pos 10: Sun   ☀️     
Pos 11: Sun   ☀️  🟢 (cause)
Pos 12: Sun   ☀️  🟢 (cause)
Pos 13: Rain  ☂  🔵 (effect)
Pos 14: Rain  ☂  🔵 (effect)
Pos 15: Sun   ☀️     
Pos 16: Rain  ☂     
Pos 17: Cloud ☁️     
Pos 18: Rain  ☂     
Pos 19: Cloud ☁️     
Pos 20: Snow  ❄️     

📊 Pattern Analysis:
   Looking for: 'Sun ☀️ → Rain ☂' with lag = 2
   🟢 First elements (cause): 3 positions
   🔵 Second elements (effect): 3 positions

   Estimated probability:
   p̂ = 3/18 = 0.167
✓ Graphic saved as 'weather_bivariate_pattern.png'
   Probability: 3/18 = 0.167

✅ File saved! You can now insert 'weather_bivariate_pattern.png' into your presentation.


C:\Users\victo\AppData\Local\Temp\ipykernel_5620\4164020674.py:137: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Weathersymbols

In [94]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.use('Agg')

# ============================================
# ORIGINAL WORKING VERSION - No font changes
# ============================================

def plot_weather_symbols(output_file='weather_symbols.png'):
    """Plot three weather symbols: Sun, Cloud, Rain with set notation"""
    
    fig, ax = plt.subplots(figsize=(8, 3.5))
    
    # Define symbols and their positions
    symbols = ['☀️', '☁️', '☔']
    labels = ['Sun', 'Cloud', 'Rain']
    colors = ['#FFD700', '#B0BEC5', '#4FC3F7']
    positions = [0, 1, 2]
    
    for i, (pos, symbol, label, color) in enumerate(zip(positions, symbols, labels, colors)):
        # Text with bbox (original working method)
        ax.text(pos, 0.2, symbol, 
                fontsize=45, ha='center', va='center',
                bbox=dict(boxstyle='circle,pad=0.3', 
                         facecolor=color, 
                         edgecolor='black', 
                         linewidth=2))
    
    # Add set notation at the bottom
    ax.text(-0.7,0.2 , r"$X_t \in S = \{$", 
            fontsize=25, ha='center', va='center')
    ax.text(0.5, 0.2, ",", 
            fontsize=25, ha='center', va='center')
    ax.text(1.5, 0.2, ",", 
            fontsize=25, ha='center', va='center')
    ax.text(2.35,0.2 , "}", 
            fontsize=25, ha='center', va='center')
    ax.set_xlim(-0.5, 2.5)
    ax.set_ylim(0.1, 0.3)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ Saved: {output_file}")

if __name__ == "__main__":
    plot_weather_symbols('../Graphs/weather_symbols.png')

✓ Saved: ../Graphs/weather_symbols.png


## Nominal / Ordinal

In [9]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import matplotlib
matplotlib.use('Agg')

# ============================================
# NOMINAL VS ORDINAL PLOT
# ============================================

def plot_nominal_vs_ordinal(output_file='nominal_ordinal.png'):
    """Compare nominal (4 symbols, no order) vs ordinal (3 symbols with order)"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # ===== LEFT: Nominal (4 symbols in circle, no order) =====
    symbols_nominal = ['☀️', '☁️', '☔', '❄️']
    colors_nominal = ['#FFD700', '#B0BEC5', '#4FC3F7', '#E0E0E0']
    
    # Circular arrangement
    angles = np.linspace(0, 2*np.pi, 4, endpoint=False)
    radius = 1.2
    
    for i, (symbol, color, angle) in enumerate(zip(symbols_nominal, colors_nominal, angles)):
        x = radius * np.cos(angle)
        y = radius * np.sin(angle)
        
        ax1.text(x, y, symbol, 
                fontsize=40, ha='center', va='center',
                bbox=dict(boxstyle='circle,pad=0.25', 
                         facecolor=color, 
                         edgecolor='black', 
                         linewidth=2))
    
    # Dashed connections to show no hierarchy
    for i in range(4):
        for j in range(i+1, 4):
            x1 = radius * np.cos(angles[i])
            y1 = radius * np.sin(angles[i])
            x2 = radius * np.cos(angles[j])
            y2 = radius * np.sin(angles[j])
            ax1.plot([x1, x2], [y1, y2], 'k--', alpha=0.3, lw=0.8)
    
    ax1.set_xlim(-1.8, 1.8)
    ax1.set_ylim(-1.8, 1.8)
    ax1.set_title("Nominal (no order)", fontsize=14, fontweight='bold', pad=10)
    ax1.set_aspect('equal')
    ax1.set_xticks([])
    ax1.set_yticks([])
    for spine in ax1.spines.values():
        spine.set_visible(False)
    
    # ===== RIGHT: Ordinal (3 symbols in line, ordered by intensity) =====
    symbols_ordinal = ['☀️', '☁️', '☔']
    colors_ordinal = ['#FFD700', '#B0BEC5', '#4FC3F7']
    ordinal_labels = ['dry', 'cloudy', 'rainy']
    positions = [0, 1, 2]
    
    for i, (pos, symbol, color, ordinal_label) in enumerate(zip(positions, symbols_ordinal, colors_ordinal, ordinal_labels)):
        ax2.text(pos, 0.2, symbol, 
                fontsize=40, ha='center', va='center',
                bbox=dict(boxstyle='circle,pad=0.25', 
                         facecolor=color, 
                         edgecolor='black', 
                         linewidth=2))
        ax2.text(pos, -0.5, ordinal_label, fontsize=9, ha='center', va='center', color='gray', style='italic')
        
        # Add ">" between categories
        if i < len(positions) - 1:
            ax2.text(pos + 0.5, 0.2, '>', fontsize=20, ha='center', va='center', color='blue', fontweight='bold')
    
    ax2.set_xlim(-0.5, 2.5)
    ax2.set_ylim(-1.2, 1)
    ax2.set_title("Ordinal (with order)", fontsize=14, fontweight='bold', pad=10)
    ax2.set_xticks([])
    ax2.set_yticks([])
    for spine in ax2.spines.values():
        spine.set_visible(False)
    
    # Add intensity arrow at bottom
    ax2.annotate('', xy=(2.3, -1.0), xytext=(-0.3, -1.0),
                arrowprops=dict(arrowstyle='<->', color='gray', lw=1.5, alpha=0.5))
    ax2.text(1, -1.1, 'increasing intensity →', fontsize=9, ha='center', color='gray')
    
    plt.suptitle("Nominal vs. Ordinal Scaling", fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ Saved: {output_file}")

if __name__ == "__main__":
    plot_nominal_vs_ordinal('nominal_ordinal.png')

✓ Saved: nominal_ordinal.png


## Mode/ Mean

In [54]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import matplotlib
matplotlib.use('Agg')

# ============================================
# MODE VS MEDIAN PLOT
# ============================================

def plot_mode_median(output_file='mode_median.png'):
    """Bar chart showing Mode (Sun) and Median (Cloud) for ordinal data"""
    
    # Ordinal categories: Sun (lowest), Cloud (middle), Rain (highest)
    categories = ['Sun ☀️', 'Cloud ☁️', 'Rain ☔']
    x = np.arange(len(categories))
    
    # Distribution where Mode = Sun (outer category) and Median = Cloud (middle)
    # Sun is most frequent (0.50), Cloud is middle, Rain is least frequent
    probs = np.array([0.40, 0.30, 0.30])
    
    # Calculate mode (highest probability)
    mode_idx = np.argmax(probs)  # Index 0 = Sun
    
    # Calculate median (where cumulative probability >= 0.5)
    cumsum = np.cumsum(probs)
    median_idx = np.where(cumsum >= 0.5)[0][0]  # Index 1 = Cloud
    
    # Colors
    colors = ['#B0BEC5' for _ in range(len(categories))]
    colors[mode_idx] = '#4CAF50'   # Mode in green
    colors[median_idx] = '#FF9800' # Median in orange
    
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # Create bars
    bars = ax.bar(x, probs, color=colors, edgecolor='black', linewidth=1.5, alpha=0.8, width=0.6)
    
    # Add value labels on bars
    for bar, prob in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
               f'{prob:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # Mark mode
    ax.annotate('Mode', xy=(mode_idx+0.2, probs[mode_idx]), xytext=(mode_idx + 0.2, probs[mode_idx] + 0.06),
               ha='center', fontsize=11, fontweight='bold', color='#4CAF50',
               arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=1.5))
    
    # Mark median
    ax.annotate('Median', xy=(median_idx+0.2, probs[median_idx]), xytext=(median_idx + 0.2, probs[median_idx] + 0.06),
               ha='center', fontsize=11, fontweight='bold', color='#FF9800',
               arrowprops=dict(arrowstyle='->', color='#FF9800', lw=1.5))
    
    
    ax.set_xlabel('Category (Ordinal Scale)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Probability', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.set_ylim(0, max(probs) + 0.12)
    
    # Add legend
    legend_elements = [
        mpatches.Patch(facecolor='#4CAF50', edgecolor='black', label='Mode '),
        mpatches.Patch(facecolor='#FF9800', edgecolor='black', label='Median')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ Saved: {output_file}")

if __name__ == "__main__":
    plot_mode_median('mode_median.png')

✓ Saved: mode_median.png


## Min/ Max Dispersion

In [105]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
import matplotlib
matplotlib.use('Agg')
from collections import Counter

# ============================================
# DISPERSION PLOTS (3 scenarios)
# ============================================

def plot_dispersion_scenarios(output_file='dispersion_scenarios.png'):
    """Plot 3 dispersion scenarios: minimal, nominal maximal, ordinal maximal"""
    
    symbols = {'Sun': '☀️', 'Cloud': '☁️', 'Rain': '☔'}
    colors = {'Sun': '#FFD700', 'Cloud': '#B0BEC5', 'Rain': '#4FC3F7'}
    categories = ['Sun', 'Cloud', 'Rain']
    x_pos = np.arange(len(categories))
    
    # Scenario 1: Minimal dispersion (all mass on Sun)
    probs_minimal = np.array([1.00, 0.00, 0.00])
    
    # Scenario 2: Nominal maximal dispersion (evenly spread)
    probs_nominal_max = np.array([0.33, 0.34, 0.33])
    
    # Scenario 3: Ordinal maximal dispersion (mass on outer categories)
    probs_ordinal_max = np.array([0.50, 0.00, 0.50])
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # ===== PLOT 1: Minimal Dispersion =====
    bars1 = axes[0].bar(x_pos, probs_minimal, color=[colors[cat] for cat in categories], 
                        edgecolor='black', linewidth=1.5, alpha=0.8, width=0.6)
    
    for bar, prob in zip(bars1, probs_minimal):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{prob:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels([f'{symbols[cat]} {cat}' for cat in categories])
    axes[0].set_ylabel('Probability', fontsize=10, fontweight='bold')
    axes[0].set_title('Minimal Dispersion\n(all mass on one category)', fontsize=11, fontweight='bold')
    axes[0].set_ylim(0, 1.1)
    axes[0].set_yticks(np.arange(0, 1.1, 0.2))
    for spine in axes[0].spines.values():
        spine.set_visible(True)
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)
    
    # Add dispersion value
    axes[0].text(0.5, 0.95, 'IQV = 0.00', transform=axes[0].transAxes,
                fontsize=9, ha='center', fontweight='bold', color='darkred')
    
    # ===== PLOT 2: Nominal Maximal Dispersion =====
    bars2 = axes[1].bar(x_pos, probs_nominal_max, color=[colors[cat] for cat in categories], 
                        edgecolor='black', linewidth=1.5, alpha=0.8, width=0.6)
    
    for bar, prob in zip(bars2, probs_nominal_max):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{prob:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels([f'{symbols[cat]} {cat}' for cat in categories])
    axes[1].set_ylabel('Probability', fontsize=10, fontweight='bold')
    axes[1].set_title('Nominal Maximal Dispersion\n(evenly spread)', fontsize=11, fontweight='bold')
    axes[1].set_ylim(0, 1.1)
    axes[1].set_yticks(np.arange(0, 1.1, 0.2))
    for spine in axes[1].spines.values():
        spine.set_visible(True)
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)
    
    # Add dispersion value
    axes[1].text(0.5, 0.95, 'IQV = 1.00', transform=axes[1].transAxes,
                fontsize=9, ha='center', fontweight='bold', color='darkgreen')
    
    # ===== PLOT 3: Ordinal Maximal Dispersion =====
    bars3 = axes[2].bar(x_pos, probs_ordinal_max, color=[colors[cat] for cat in categories], 
                        edgecolor='black', linewidth=1.5, alpha=0.8, width=0.6)
    
    for bar, prob in zip(bars3, probs_ordinal_max):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{prob:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    axes[2].set_xticks(x_pos)
    axes[2].set_xticklabels([f'{symbols[cat]} {cat}' for cat in categories])
    axes[2].set_ylabel('Probability', fontsize=10, fontweight='bold')
    axes[2].set_title('Ordinal Maximal Dispersion\n(mass on outer categories)', fontsize=11, fontweight='bold')
    axes[2].set_ylim(0, 1.1)
    axes[2].set_yticks(np.arange(0, 1.1, 0.2))
    for spine in axes[2].spines.values():
        spine.set_visible(True)
    axes[2].spines['top'].set_visible(False)
    axes[2].spines['right'].set_visible(False)
    
    # Add dispersion value
    axes[2].text(0.5, 0.95, 'IOV = 1.00', transform=axes[2].transAxes,
                fontsize=9, ha='center', fontweight='bold', color='darkgreen')
    
 
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ Saved: {output_file}")
    
    return output_file

# ============================================
# ALTERNATIVE: Separate plots
# ============================================

def plot_minimal_dispersion(output_file='minimal_dispersion.png'):
    """Plot minimal dispersion (all mass on one category)"""
    
    symbols = {'Sun': '☀️', 'Cloud': '☁️', 'Rain': '☔'}
    colors = {'Sun': '#FFD700', 'Cloud': '#B0BEC5', 'Rain': '#4FC3F7'}
    categories = ['Sun', 'Cloud', 'Rain']
    x_pos = np.arange(len(categories))
    probs = np.array([1.00, 0.00, 0.00])
    
    fig, ax = plt.subplots(figsize=(6, 4))
    
    bars = ax.bar(x_pos, probs, color=[colors[cat] for cat in categories], 
                  edgecolor='black', linewidth=1.5, alpha=0.8, width=0.6)
    
    for bar, prob in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
               f'{prob:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'{symbols[cat]} {cat}' for cat in categories])
    ax.set_ylabel('Probability', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.1)
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ Saved: {output_file}")

def plot_nominal_maximal_dispersion(output_file='nominal_maximal_dispersion.png'):
    """Plot nominal maximal dispersion (evenly spread)"""
    
    symbols = {'Sun': '☀️', 'Cloud': '☁️', 'Rain': '☔'}
    colors = {'Sun': '#FFD700', 'Cloud': '#B0BEC5', 'Rain': '#4FC3F7'}
    categories = ['Sun', 'Cloud', 'Rain']
    x_pos = np.arange(len(categories))
    probs = np.array([0.33, 0.34, 0.33])
    
    fig, ax = plt.subplots(figsize=(6, 4))
    
    bars = ax.bar(x_pos, probs, color=[colors[cat] for cat in categories], 
                  edgecolor='black', linewidth=1.5, alpha=0.8, width=0.6)
    
    for bar, prob in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
               f'{prob:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'{symbols[cat]} {cat}' for cat in categories])
    ax.set_ylabel('Probability', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.1)
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ Saved: {output_file}")

def plot_ordinal_maximal_dispersion(output_file='ordinal_maximal_dispersion.png'):
    """Plot ordinal maximal dispersion (mass on outer categories)"""
    
    symbols = {'Sun': '☀️', 'Cloud': '☁️', 'Rain': '☔'}
    colors = {'Sun': '#FFD700', 'Cloud': '#B0BEC5', 'Rain': '#4FC3F7'}
    categories = ['Sun', 'Cloud', 'Rain']
    x_pos = np.arange(len(categories))
    probs = np.array([0.50, 0.00, 0.50])
    
    fig, ax = plt.subplots(figsize=(6, 4))
    
    bars = ax.bar(x_pos, probs, color=[colors[cat] for cat in categories], 
                  edgecolor='black', linewidth=1.5, alpha=0.8, width=0.6)
    
    for bar, prob in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
               f'{prob:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'{symbols[cat]} {cat}' for cat in categories])
    ax.set_ylabel('Probability', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.1)
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ Saved: {output_file}")

if __name__ == "__main__":
    # Create combined plot
    plot_dispersion_scenarios('dispersion_scenarios.png')
    
    # Or create separate plots
    plot_minimal_dispersion('../Graphs/minimal_dispersion.png')
    plot_nominal_maximal_dispersion('../Graphs/nominal_maximal_dispersion.png')
    plot_ordinal_maximal_dispersion('../Graphs/ordinal_maximal_dispersion.png')

✓ Saved: dispersion_scenarios.png
✓ Saved: ../Graphs/minimal_dispersion.png
✓ Saved: ../Graphs/nominal_maximal_dispersion.png
✓ Saved: ../Graphs/ordinal_maximal_dispersion.png


## Marginal Estimation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
import matplotlib
matplotlib.use('Agg')
from collections import Counter
import random

# ============================================
# MARGINAL ESTIMATION PLOT (3 subplots)
# ============================================

def plot_marginal_estimation(output_file='marginal_estimation.png'):
    """Visualize marginal estimation using 3 ordinal categories: Sun, Cloud, Rain"""
    
    # Create sample sequence with 3 ordinal categories
    random.seed(42)
    sequence = ['Sun', 'Cloud', 'Sun', 'Rain', 'Cloud', 'Sun', 'Rain', 'Cloud', 
                'Rain', 'Sun', 'Cloud', 'Sun', 'Rain', 'Cloud', 'Sun', 'Rain',
                'Cloud', 'Sun', 'Rain', 'Sun']
    
    symbols = {'Sun': '☀️', 'Cloud': '☁️', 'Rain': '☔'}
    colors = {'Sun': '#FFD700', 'Cloud': '#B0BEC5', 'Rain': '#4FC3F7'}
    
    # Create figure with GridSpec: 1 row, 2 columns, but right column has 2 rows
    fig = plt.figure(figsize=(14, 5))
    gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1], width_ratios=[1, 1])
    
    # Left plot (spanning both rows)
    ax1 = plt.subplot(gs[:, 0])
    
    # Right plots
    ax2 = plt.subplot(gs[0, 1])  # Top-right (PMF)
    ax3 = plt.subplot(gs[1, 1])  # Bottom-right (CDF)
    
    # ===== LEFT: Sequence visualization =====
    for i, weather in enumerate(sequence):
        ax1.text(i, 0, symbols[weather], 
                fontsize=22, ha='center', va='center',
                bbox=dict(boxstyle='circle,pad=0.18', 
                         facecolor=colors[weather], 
                         edgecolor='black', 
                         linewidth=1.5))
        ax1.text(i, -0.6, str(i+1), fontsize=7, ha='center', va='center', color='gray')
    
    ax1.set_xlim(-0.5, len(sequence) - 0.5)
    ax1.set_ylim(-1, 1)
    ax1.set_title('Observed Time Series $(X_1, \\ldots, X_n)$', fontsize=11, fontweight='bold', pad=2)
    ax1.set_xlabel('Time $t$', fontsize=9)
    ax1.set_xticks([])
    ax1.set_yticks([])
    for spine in ax1.spines.values():
        spine.set_visible(False)
    
    # ===== TOP-RIGHT: PMF (Count frequencies) =====
    counts = Counter(sequence)
    categories = ['Sun', 'Cloud', 'Rain']
    frequencies = [counts[cat] / len(sequence) for cat in categories]
    x_pos = np.arange(len(categories))
    
    bars = ax2.bar(x_pos, frequencies, color=[colors[cat] for cat in categories], 
                  edgecolor='black', linewidth=1.5, alpha=0.8, width=0.6)
    
    # Add count labels on bars
    for bar, cat in zip(bars, categories):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{bar.get_height():.2f}', 
                ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels([f'{symbols[cat]} {cat}' for cat in categories])
    ax2.set_ylabel('Relative Frequency', fontsize=9, fontweight='bold')
    ax2.set_ylim(0, max(frequencies) * 3)
    for spine in ax2.spines.values():
        spine.set_visible(True)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    
    # ===== BOTTOM-RIGHT: CDF =====
    cdf = np.cumsum(frequencies)
    
    bars_cdf = ax3.bar(x_pos, cdf, color=[colors[cat] for cat in categories], 
                       edgecolor='black', linewidth=1.5, alpha=0.7, width=0.6)
    
    # Add value labels on bars
    for bar, val in zip(bars_cdf, cdf):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels([f'{symbols[cat]} {cat}' for cat in categories])
    ax3.set_ylabel('Cumulative Probability', fontsize=9, fontweight='bold')
    ax3.set_xlabel('Category', fontsize=9)
    ax3.set_ylim(0, max(frequencies) * 3)
    for spine in ax3.spines.values():
        spine.set_visible(True)
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ Saved: {output_file}")

if __name__ == "__main__":
    plot_marginal_estimation('marginal_estimation.png')

✓ Saved: marginal_estimation2.png


## Bivariate Probabilities

In [74]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import matplotlib
matplotlib.use('Agg')
import random

# ============================================
# BIVARIATE PROBABILITY MATRIX WITH FULL NOTATION
# ============================================

def plot_bivariate_matrix(output_file='bivariate_matrix.png', lag=2):
    """Visualize bivariate probability matrix with full notation in each cell"""
    
    # Define categories and symbols
    categories = ['Sun', 'Cloud', 'Rain']
    symbols = ['☀️', '☁️', '☔']
    n_cat = len(categories)
    
    # Create synthetic bivariate probability matrix
    bivariate_probs = np.array([
        [0.35, 0.10, 0.05],  # Sun followed by Sun, Cloud, Rain
        [0.08, 0.20, 0.07],  # Cloud followed by Sun, Cloud, Rain
        [0.04, 0.06, 0.05]   # Rain followed by Sun, Cloud, Rain
    ])
    
    # Normalize to ensure sum = 1
    bivariate_probs = bivariate_probs / bivariate_probs.sum()
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Create heatmap
    im = ax.imshow(bivariate_probs, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.4)
    
    # Add text annotations with full notation in each cell
    for i in range(n_cat):
        for j in range(n_cat):
            # Create full notation text
            notation = f"$P(X_t = {symbols[i]},\\ X_{{t+{lag}}} = {symbols[j]})$\n= {bivariate_probs[i, j]:.3f}"
            
            text = ax.text(j, i, notation,
                          ha="center", va="center", 
                          color="black" if bivariate_probs[i, j] < 0.25 else "white",
                          fontsize=16, fontweight='bold')
    
    # Set ticks and labels with symbols only
    ax.set_xticks(np.arange(n_cat))
    ax.set_yticks(np.arange(n_cat))
    ax.set_xticklabels([f'{symbols[i]}' for i in range(n_cat)], fontsize=20, fontweight='bold')
    ax.set_yticklabels([f'{symbols[i]}' for i in range(n_cat)], fontsize=20, fontweight='bold')
    
    # Add axis labels
    ax.set_xlabel(f'$X_{{t+{lag}}}$ (Future, lag={lag})', fontsize=14, fontweight='bold', labelpad=15)
    ax.set_ylabel('$X_t$ (Current)', fontsize=14, fontweight='bold', labelpad=15)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Probability', fontsize=11, fontweight='bold')
    
    # Add grid lines
    for i in range(n_cat + 1):
        ax.axhline(i - 0.5, color='white', linewidth=2)
        ax.axvline(i - 0.5, color='white', linewidth=2)
    
    # Add marginal probabilities on sides
    row_marginals = bivariate_probs.sum(axis=1)
    col_marginals = bivariate_probs.sum(axis=0)
    


    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ Saved: {output_file}")
    
    # Print additional info
    print(f"\nBivariate Probability Matrix at Lag {lag}:")
    print("="*50)
    print(f"Row sums (P(X_t = i)): {row_marginals}")
    print(f"Column sums (P(X_{{t+{lag}}} = j)): {col_marginals}")
    print(f"Total sum: {bivariate_probs.sum():.3f}")

if __name__ == "__main__":
    plot_bivariate_matrix('bivariate_matrix.png', lag=2)

✓ Saved: bivariate_matrix.png

Bivariate Probability Matrix at Lag 2:
Row sums (P(X_t = i)): [0.5  0.35 0.15]
Column sums (P(X_{t+2} = j)): [0.47 0.36 0.17]
Total sum: 1.000


## Bivariate Estimates

In [2]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import matplotlib
matplotlib.use('Agg')
import random

# ============================================
# CONFIRMATION PROBABILITIES WITH MISSINGNESS (Lag-2)
# P(X_t = Sun, X_{t+h} = Sun)
# ============================================

def plot_confirmation_probabilities(output_file='confirmation_probabilities.png', lag=2):
    """Visualize confirmation probabilities P(X_t = Sun, X_{t+h} = Sun) with missingness"""
    
    # Create sequence with 3 ordinal categories
    random.seed(42)
    symbols = {'Sun': '☀️', 'Cloud': '☁️', 'Rain': '☔'}
    colors = {'Sun': '#FFD700', 'Cloud': '#B0BEC5', 'Rain': '#4FC3F7'}
    
    # Create longer sequence
    sequence = ['Sun', 'Sun', 'Cloud', 'Rain', 'Sun', 'Cloud', 'Rain', 'Sun', 
                'Cloud', 'Sun', 'Sun', 'Cloud', 'Rain', 'Sun', 'Cloud', 'Sun',
                'Rain', 'Cloud', 'Sun', 'Rain', 'Sun', 'Sun', 'Cloud', 'Rain',
                'Sun', 'Cloud', 'Rain', 'Sun', 'Sun', 'Cloud']
    
    n = len(sequence)
    
    # Missingness pattern (MCAR: 25% missing)
    # 1 = observed, 0 = missing
    missing_pattern = [1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 
                       1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1]
    
    # Find confirmation patterns: Sun at t AND Sun at t+lag (both observed)
    confirmation_indices = []  # Store (i, i+lag) for confirmation pairs
    first_highlight = [False] * n
    second_highlight = [False] * n
    missing_positions = [False] * n
    
    for i in range(n - lag):
        # Check if both positions are observed
        if missing_pattern[i] == 1 and missing_pattern[i + lag] == 1:
            if sequence[i] == 'Sun' and sequence[i + lag] == 'Sun':
                confirmation_indices.append((i, i + lag))
                first_highlight[i] = True
                second_highlight[i + lag] = True
    
    # Count confirmation probabilities
    n_possible_pairs = 0
    n_confirmation_pairs = 0
    
    for i in range(n - lag):
        if missing_pattern[i] == 1 and missing_pattern[i + lag] == 1:
            n_possible_pairs += 1
            if sequence[i] == 'Sun' and sequence[i + lag] == 'Sun':
                n_confirmation_pairs += 1
    
    for i in range(n):
        if missing_pattern[i] == 0:
            missing_positions[i] = True
    
    confirmation_prob = n_confirmation_pairs / n_possible_pairs if n_possible_pairs > 0 else 0
    
    # Create figure with 2 subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
    
    # ===== LEFT: Time series with missingness and confirmation patterns =====
    for i, weather in enumerate(sequence):
        if missing_positions[i]:
            # Missing - greyed out with '?'
            rect = plt.Rectangle((i - 0.4, -0.7), 0.8, 1.0, 
                                 facecolor='lightgray', alpha=0.6, 
                                 edgecolor='gray', linewidth=1.5, 
                                 linestyle='--', zorder=0)
            ax1.add_patch(rect)
            ax1.text(i, 0, '?', 
                    fontsize=18, ha='center', va='center', color='red',
                    bbox=dict(boxstyle='circle,pad=0.15', 
                             facecolor='lightgray', 
                             edgecolor='gray', 
                             linewidth=1))
        else:
            # Observed - normal
            # Highlight confirmation pattern roles
            if first_highlight[i]:
                rect = plt.Rectangle((i - 0.4, -0.7), 0.8, 1.0, 
                                     facecolor='lightgreen', alpha=0.5, 
                                     edgecolor='green', linewidth=2, zorder=0)
                ax1.add_patch(rect)
            elif second_highlight[i]:
                rect = plt.Rectangle((i - 0.4, -0.7), 0.8, 1.0, 
                                     facecolor='lightblue', alpha=0.5, 
                                     edgecolor='blue', linewidth=2, zorder=0)
                ax1.add_patch(rect)
            
            ax1.text(i, 0, symbols[weather], 
                    fontsize=18, ha='center', va='center',
                    bbox=dict(boxstyle='circle,pad=0.15', 
                             facecolor=colors[weather], 
                             edgecolor='black', 
                             linewidth=1.2))
        
        ax1.text(i, -0.65, str(i+1), fontsize=7, ha='center', va='center', color='gray')
    
    # Add connecting arrows for confirmation patterns
    for i, j in confirmation_indices:
        ax1.annotate('', xy=(j, 0.35), xytext=(i, 0.35),
                    arrowprops=dict(arrowstyle='->', color='darkgreen', 
                                   lw=2, alpha=0.8))
        mid_x = i + lag/2
        ax1.text(mid_x, 0.48, f'lag={lag}', fontsize=7, ha='center',
                color='darkgreen', fontweight='bold')
    
    ax1.set_xlim(-0.5, n - 0.5)
    ax1.set_ylim(-1, 0.8)
    ax1.set_title(f'Observed Time Series with Confirmation Patterns\n(Sun → Sun at lag={lag})', 
                  fontsize=11, fontweight='bold', pad=10)
    ax1.set_xlabel('Time $t$', fontsize=9)
    ax1.set_ylabel('$O_t X_t$', fontsize=9)
    ax1.set_xticks([])
    ax1.set_yticks([])
    for spine in ax1.spines.values():
        spine.set_visible(False)
    
    # Add legend for left plot
    legend_elements = [
        mpatches.Patch(facecolor='lightgray', edgecolor='gray', alpha=0.6, 
                      label='Missing observation'),
        mpatches.Patch(facecolor='lightgreen', edgecolor='green', alpha=0.5, 
                      label='First element: Sun (cause)'),
        mpatches.Patch(facecolor='lightblue', edgecolor='blue', alpha=0.5, 
                      label=f'Second element: Sun (effect at lag={lag})'),
        plt.Line2D([0], [0], color='darkgreen', lw=2, label='Confirmation pattern')
    ]
    ax1.legend(handles=legend_elements, loc='upper left', fontsize=7, framealpha=0.9)
    
    # ===== RIGHT: Confirmation probability estimation =====
    categories = ['Sun', 'Cloud', 'Rain']
    x_pos = np.arange(len(categories))
    
    # Confirmation probability matrix (Sun-Sun highlighted)
    confirmation_matrix = np.zeros((3, 3))
    total_pairs = np.zeros((3, 3))
    
    for i in range(n - lag):
        if missing_pattern[i] == 1 and missing_pattern[i + lag] == 1:
            row = categories.index(sequence[i])
            col = categories.index(sequence[i + lag])
            total_pairs[row, col] += 1
            confirmation_matrix[row, col] += 1
    
    # Convert to probabilities
    with np.errstate(divide='ignore', invalid='ignore'):
        bivariate_probs = np.divide(confirmation_matrix, total_pairs, 
                                    where=total_pairs > 0)
        bivariate_probs[total_pairs == 0] = 0
    
    # Create heatmap
    im = ax2.imshow(bivariate_probs, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.6)
    
    # Add text annotations
    for i in range(3):
        for j in range(3):
            if total_pairs[i, j] > 0:
                text = f'{bivariate_probs[i, j]:.3f}\n(n={int(total_pairs[i, j])})'
            else:
                text = '-\nn=0'
            ax2.text(j, i, text, ha="center", va="center", 
                    color="black" if bivariate_probs[i, j] < 0.35 else "white",
                    fontsize=9, fontweight='bold')
    
    # Set ticks and labels
    ax2.set_xticks(np.arange(3))
    ax2.set_yticks(np.arange(3))
    ax2.set_xticklabels([f'{symbols[cat]}\n{cat}' for cat in categories], fontsize=10)
    ax2.set_yticklabels([f'{symbols[cat]}\n{cat}' for cat in categories], fontsize=10)
    
    ax2.set_xlabel(f'$X_{{t+{lag}}}$ (Future, lag={lag})', fontsize=11, fontweight='bold', labelpad=10)
    ax2.set_ylabel('$X_t$ (Current)', fontsize=11, fontweight='bold', labelpad=10)
    ax2.set_title(f'Estimated Bivariate Probabilities (with Missingness)\n$\\hat{{p}}_{{ij}}({lag}) = n_{{ij}} / n_{{obs}}$', 
                  fontsize=10, fontweight='bold', pad=10)
    
    # Add grid lines
    for i in range(4):
        ax2.axhline(i - 0.5, color='white', linewidth=2)
        ax2.axvline(i - 0.5, color='white', linewidth=2)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax2, shrink=0.7)
    cbar.set_label('Probability', fontsize=9, fontweight='bold')
    
    # Highlight confirmation cell (Sun → Sun)
    sun_idx = categories.index('Sun')
    rect = plt.Rectangle((sun_idx - 0.5, sun_idx - 0.5), 1, 1, 
                         fill=False, edgecolor='darkgreen', linewidth=3, linestyle='--')
    ax2.add_patch(rect)
    
    # Formula box for confirmation probability
    formula = f"$\\hat{{p}}_{{\\text{{conf}}}} = P(X_t = ☀️,\\ X_{{t+{lag}}} = ☀️) = \\frac{{{n_confirmation_pairs}}}{{{n_possible_pairs}}} = {confirmation_prob:.3f}$"
    ax2.text(0.5, -0.55, formula, fontsize=9, ha='center', va='center',
            transform=ax2.transAxes,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', 
                     edgecolor='darkgreen', linewidth=1.5))
    
    plt.suptitle(f'Confirmation Probabilities P(X_t = ☀️, X_{{t+{lag}}} = ☀️) with MCAR (25% missing)', 
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"✓ Saved: {output_file}")
    print(f"   Confirmation probability P(Sun → Sun) at lag={lag}: {confirmation_prob:.3f}")
    print(f"   Observed pairs: {n_confirmation_pairs} / {n_possible_pairs}")
    
    return output_file

if __name__ == "__main__":
    plot_confirmation_probabilities('confirmation_probabilities.png', lag=2)

✓ Saved: confirmation_probabilities.png
   Confirmation probability P(Sun → Sun) at lag=2: 0.143
   Observed pairs: 2 / 14
